# 33 - Validação e resumo da Semana 3

| | |
|---|---|
| **Tarefa do plano** | S3-04 - relatório de validação (IoU, F1, Kappa, ROC-AUC) |
| **O que faz** | Junta num só sítio os resultados dos modelos e dos alertas, com o que está provado e o que não está |
| **Usa** | Tabelas produzidas pelos notebooks 30, 31 e 32 |
| **Produz** | `reports/semana_3/33_relatorio_de_validacao.csv` |

**Como ler as medidas** (todas entre 0 e 1; quanto mais alto, melhor)

| Medida | Pergunta a que responde |
|---|---|
| IoU | da área que o modelo **ou** a referência dizem ser água, que parte é comum aos dois? |
| F1 | equilíbrio entre não falhar água e não dar falsos alarmes |
| Kappa | concordância com a referência, descontando o que se acertaria por acaso |
| Precisão | do que o modelo chamou água, que parte era mesmo água? |
| Recall | da água que existia, que parte o modelo encontrou? |
| AUC | o modelo dá valores mais altos à água do que à não-água? (0,5 = acaso) |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ete import alerts, config, drivers, rasters
from ete import grid as G
from ete import models as M

cfg = config.load_config()
P = cfg["processing"]
REPORTS = config.path(cfg, "reports", "semana_3")       # figuras e tabelas para o relatório
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Modelo B (deteção de água e cheia, 10 m)

In [2]:
b = pd.read_csv(REPORTS / "30_modelo_b_algoritmos.csv", index_col=0)
b_features = pd.read_csv(REPORTS / "30_modelo_b_features.csv", index_col=0)
display(b_features[["iou", "f1", "kappa", "precisao", "recall", "auc"]])
b[["iou", "f1", "kappa", "precisao", "recall", "auc", "iou_desvio_entre_blocos", "segundos"]]

,iou,f1,kappa,precisao,recall,auc
features,,,,,,
só JRC (radar),0.639,0.780,0.584,0.989,0.644,0.968
radar,0.899,0.947,0.876,0.985,0.912,0.982
radar + terreno,0.906,0.951,0.884,0.989,0.915,0.987
radar + ótico,0.926,0.962,0.908,0.982,0.942,0.988
regra simples (VH p10 <= -22 dB),0.880,0.936,0.842,0.937,0.935,NaN


,iou,f1,kappa,precisao,recall,auc,iou_desvio_entre_blocos,segundos
algoritmo,,,,,,,,
Regressão logística,0.791,0.883,0.744,0.970,0.810,0.953,0.031,3
Random Forest,0.899,0.947,0.876,0.985,0.912,0.982,0.015,28
Gradient Boosting (HistGB),0.907,0.951,0.885,0.985,0.919,0.983,0.013,20
XGBoost,0.908,0.952,0.886,0.985,0.920,0.983,0.015,6


## 2. Modelo A (suscetibilidade, 30 m)

In [3]:
a = pd.read_csv(REPORTS / "31_modelo_a_algoritmos.csv", index_col=0)
display(pd.read_csv(REPORTS / "31_modelo_a_features.csv", index_col=0))
display(a)
pd.read_csv(REPORTS / "31_modelo_a_teste_2026.csv", index_col=0)

,auc_validacao_2020_21,auc_cheia_2026
features,,
terreno,0.959,0.950
terreno sem distâncias à água,0.932,0.920
"terreno + chuva, caudal e humidade",0.982,0.966
só HAND (modelo),0.782,0.798
"HAND sozinho, sem modelo",0.656,0.765


,auc_validacao_2020_21,auc_cheia_2026,segundos
algoritmo,,,
Regressão logística,0.853,0.910,2
Random Forest,0.959,0.950,9
Gradient Boosting (HistGB),0.962,0.959,5
XGBoost,0.962,0.960,3


,area_com_referencia_km2,parte_da_classe_que_alagou,parte_da_cheia_nesta_classe
baixa,110.027,0.003,0.001
média,179.708,0.565,0.352
alta,153.796,0.955,0.508
muito alta,40.691,0.984,0.139


## 3. Alertas

In [4]:
display(pd.read_csv(REPORTS / "32_antecedencia_dos_alertas.csv", index_col=0))
pd.read_csv(REPORTS / "32_exposicao_por_classe.csv", index_col=0)

,primeiro_alerta,dias_de_antecedencia,nivel_no_dia
jan 2026: primeira cheia mapeada em Marracuene (Copernicus),2026-01-12,5,vermelho
jan 2026: maior extensão mapeada (Copernicus),2026-01-12,12,vermelho
fev 2023: ativação do Copernicus para a província de Maputo,2023-02-10,1,vermelho


,area_km2,area_alagada_km2,edificios,edificios_em_zona_alagada,escolas,escolas_em_zona_alagada,unidades_de_saude,unidades_de_saude_em_zona_alagada
suscetibilidade,,,,,,,,
baixa,282.300,0.200,148391,14,12,0,12,0
média,233.500,82.400,52336,299,2,0,3,0
alta,122.800,103.800,505,56,0,0,0,0
muito alta,39.700,27.700,70,11,0,0,0,0


## 4. Relatório de validação

In [5]:
best_b = (REPORTS / "30_modelo_b_escolhido.txt").read_text().strip()
best_a = (REPORTS / "31_modelo_a_escolhido.txt").read_text().strip()
rule = b_features[b_features.index.str.startswith("regra simples")].iloc[0]
mark = lambda name, best: f"{name} (escolhido)" if name == best else name
rows = [{"modelo": "B - " + mark(n, best_b), "teste": "cheia jan 2026, blocos espaciais de 3 km (Copernicus EMS)",
         "IoU": r["iou"], "F1": r["f1"], "Kappa": r["kappa"], "ROC-AUC": r["auc"]} for n, r in b.iterrows()]
rows.append({"modelo": "B - regra simples VH", "teste": "cheia jan 2026 (Copernicus EMS)", "IoU": rule["iou"], "F1": rule["f1"], "Kappa": rule["kappa"], "ROC-AUC": np.nan})
rows.append({"modelo": "B - " + mark(best_b, best_b), "teste": "água JRC de controlo 2016-2021", "IoU": np.nan, "F1": np.nan, "Kappa": np.nan, "ROC-AUC": b.loc[best_b, "auc_agua_jrc"]})
for n, r in a.iterrows():
    rows.append({"modelo": "A - " + mark(n, best_a), "teste": "validação temporal 2020-2021 (JRC)", "IoU": np.nan, "F1": np.nan, "Kappa": np.nan, "ROC-AUC": r["auc_validacao_2020_21"]})
    rows.append({"modelo": "A - " + mark(n, best_a), "teste": "cheia jan 2026 (Copernicus EMS)", "IoU": np.nan, "F1": np.nan, "Kappa": np.nan, "ROC-AUC": r["auc_cheia_2026"]})
report = pd.DataFrame(rows).set_index(["modelo", "teste"])
report.round(3).to_csv(REPORTS / "33_relatorio_de_validacao.csv")
print(f"Modelo B escolhido: {best_b} | Modelo A escolhido: {best_a}")
report.round(3)

Modelo B escolhido: XGBoost | Modelo A escolhido: XGBoost


IoU  \
modelo                         teste                                                      
B - Regressão logística        cheia jan 2026, blocos espaciais de 3 km (Coper... 0.791   
B - Random Forest              cheia jan 2026, blocos espaciais de 3 km (Coper... 0.899   
B - Gradient Boosting (HistGB) cheia jan 2026, blocos espaciais de 3 km (Coper... 0.907   
B - XGBoost (escolhido)        cheia jan 2026, blocos espaciais de 3 km (Coper... 0.908   
B - regra simples VH           cheia jan 2026 (Copernicus EMS)                    0.880   
B - XGBoost (escolhido)        água JRC de controlo 2016-2021                       NaN   
A - Regressão logística        validação temporal 2020-2021 (JRC)                   NaN   
                               cheia jan 2026 (Copernicus EMS)                      NaN   
A - Random Forest              validação temporal 2020-2021 (JRC)                   NaN   
                               cheia jan 2026 (Copernicus EMS)                      NaN   
A - Gradient Boosting (HistGB) validação temporal 2020-2021 (JRC)                   NaN   
                               cheia jan 2026 (Copernicus EMS)                      NaN   
A - XGBoost (escolhido)        validação temporal 2020-2021 (JRC)                   NaN   
                               cheia jan 2026 (Copernicus EMS)                      NaN   

                                                                                     F1  \
modelo                         teste                                                      
B - Regressão logística        cheia jan 2026, blocos espaciais de 3 km (Coper... 0.883   
B - Random Forest              cheia jan 2026, blocos espaciais de 3 km (Coper... 0.947   
B - Gradient Boosting (HistGB) cheia jan 2026, blocos espaciais de 3 km (Coper... 0.951   
B - XGBoost (escolhido)        cheia jan 2026, blocos espaciais de 3 km (Coper... 0.952   
B - regra simples VH           cheia jan 2026 (Copernicus EMS)                    0.936   
B - XGBoost (escolhido)        água JRC de controlo 2016-2021                       NaN   
A - Regressão logística        validação temporal 2020-2021 (JRC)                   NaN   
                               cheia jan 2026 (Copernicus EMS)                      NaN   
A - Random Forest              validação temporal 2020-2021 (JRC)                   NaN   
                               cheia jan 2026 (Copernicus EMS)                      NaN   
A - Gradient Boosting (HistGB) validação temporal 2020-2021 (JRC)                   NaN   
                               cheia jan 2026 (Copernicus EMS)                      NaN   
A - XGBoost (escolhido)        validação temporal 2020-2021 (JRC)                   NaN   
                               cheia jan 2026 (Copernicus EMS)                      NaN   

                                                                                   Kappa  \
modelo                         teste                                                       
B - Regressão logística        cheia jan 2026, blocos espaciais de 3 km (Coper...  0.744   
B - Random Forest              cheia jan 2026, blocos espaciais de 3 km (Coper...  0.876   
B - Gradient Boosting (HistGB) cheia jan 2026, blocos espaciais de 3 km (Coper...  0.885   
B - XGBoost (escolhido)        cheia jan 2026, blocos espaciais de 3 km (Coper...  0.886   
B - regra simples VH           cheia jan 2026 (Copernicus EMS)                     0.842   
B - XGBoost (escolhido)        água JRC de controlo 2016-2021                        NaN   
A - Regressão logística        validação temporal 2020-2021 (JRC)                    NaN   
                               cheia jan 2026 (Copernicus EMS)                       NaN   
A - Random Forest              validação temporal 2020-2021 (JRC)                    NaN   
                               cheia jan 2026 (Copernicus EMS)                       NaN   
A - Gradient Boosting (HistGB) validação temporal 2020-2021 (JRC) 

O Modelo A só tem AUC porque produz uma nota de risco, não um mapa de "água /
não água": IoU, F1 e Kappa não se aplicam a uma nota contínua.

## 5. O que está provado e o que não está

| Afirmação | Estado | Base |
|---|---|---|
| O Modelo B encontra a cheia de janeiro de 2026 em sítios que não viu | provado, um evento | blocos espaciais, referência Copernicus |
| O Modelo B é melhor do que uma regra simples de radar | sim, por pouco | mesma referência |
| O algoritmo escolhido é o melhor dos quatro testados | empate técnico entre os três de árvores | diferença menor do que a variação entre blocos |
| O Modelo B funciona noutras cheias | **por provar** | não há outra cheia com referência |
| O mapa de suscetibilidade ordena bem o risco | provado, um evento | cheia de 2026, que o modelo nunca viu |
| As regras de alerta teriam avisado antes da cheia de 2026 | sim, com dados de reanálise | limites calculados até 2021 |
| As regras funcionam em tempo real | **por provar** | falta ligar à previsão do GloFAS e ao IMERG |
| Taxa de falsos alarmes | **desconhecida** | não há lista oficial de cheias |
| O indicador de turbidez (NDTI) segue o caudal e a estação | sim | 116 meses de Sentinel-2, notebook 34 |
| O indicador de turbidez corresponde a valores reais (NTU) | **por provar** | faltam amostras recolhidas no rio |
| O indicador de clorofila (NDCI) mede algas | **não**, neste rio | fica dominado pelos sedimentos |

## 6. O que fica pronto para a plataforma (Semana 4)

| Produto | Ficheiro |
|---|---|
| Mapa de suscetibilidade (30 m) | `data/processed/maps/modelo_a/suscetibilidade_marracuene_30m.tif` |
| Mapas mensais de água e cheia (10 m) | `data/processed/maps/modelo_b/agua_marracuene_<mês>_10m.tif` |
| Polígonos da cheia de janeiro de 2026 | `data/processed/maps/modelo_b/cheia_marracuene_2026-01.gpkg` |
| Alertas diários 2006–2026 | `data/processed/series/alertas_diarios.csv` |
| Área de cheia por mês | `data/processed/series/modelo_b_area_mensal.csv` |
| Modelos treinados | `models/modelo_a_terreno.joblib`, `models/modelo_b_radar.joblib` |
| Exposição por classe de risco | `reports/semana_3/32_exposicao_por_classe.csv` |
| Indicadores de qualidade da água por troço e mês | `data/processed/series/qualidade_agua_mensal.csv` |